In [89]:
import pandas as pd
import numpy as np


train_df = pd.read_csv("../data/processed_data/train.csv")
test_df = pd.read_csv("../data/processed_data/test.csv")
ports_df = pd.read_csv("../data/original_data/ports.csv", sep='|')

train_df.head()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,day_of_the_week,hour_of_the_day,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,0.666667,0.608696,1,1,0,0,0,0,0,0
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,0.666667,0.608696,1,1,0,0,0,0,0,0
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,0.666667,0.608696,1,1,0,0,0,0,0,0
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,0.666667,0.652174,1,1,0,0,0,0,0,0
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,0.666667,0.652174,1,0,0,0,0,0,0,0


In [90]:
test_df.head()

,ID,vesselId,time,scaling_factor,port_lat,port_long,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
0,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.975528,0.654508,1,1,0,0,0,0,0,0
1,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.206107,1,1,1,0,0,0,0,0
2,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.345492,0.024472,1,0,0,0,0,0,0,0
3,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.793893,1,0,0,0,0,0,0,0
4,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.654508,0.975528,1,1,0,0,0,0,0,0


In [91]:
train_df["portId"].head()

0    61d376b393c6feb83e5eb50c
1    61d376d893c6feb83e5eb546
2    61d376d893c6feb83e5eb546
3    61d376d893c6feb83e5eb546
4    61d376d893c6feb83e5eb546
Name: portId, dtype: object

In [92]:
ports_df.head()

,portId,name,portLocation,longitude,latitude,UN_LOCODE,countryName,ISO
0,61d36ed80a1807568ff9a064,Port of Algiers,Algiers,3.067222,36.773611,DZALG,Algeria,DZ
1,61d36ed80a1807568ff9a065,Port of Annaba,Annaba,7.772500,36.900556,DZAAE,Algeria,DZ
2,61d36edf0a1807568ff9a070,Port of Oran,Oran,-0.639722,35.712222,DZORN,Algeria,DZ
3,61d36ee00a1807568ff9a072,Port of Skikda,Skikda,6.905833,36.887500,DZSKI,Algeria,DZ
4,61d36ee10a1807568ff9a074,Port of Pago-Pago,Pago-Pago,-170.690556,-14.274167,ASPPG,American Samoa,AS


In [93]:
def add_port_coordinates(df, ports_df):
    """
    Add port latitude and longitude columns to the input dataframe by mapping from ports_df.
    
    Parameters:
    df (pandas.DataFrame): Input dataframe containing portId column
    ports_df (pandas.DataFrame): Ports reference dataframe containing portId, latitude, and longitude
    
    Returns:
    pandas.DataFrame: Original dataframe with two new columns: port_lat and port_long
    """
    # Create a mapping dictionary for faster lookup
    port_coords = ports_df.set_index('portId')[['latitude', 'longitude']].to_dict('index')
    
    # Initialize new columns
    df['port_lat'] = float('nan')
    df['port_long'] = float('nan')
    
    # Update values for rows with valid portId
    for port_id, coords in port_coords.items():
        mask = df['portId'] == port_id
        df.loc[mask, 'port_lat'] = coords['latitude']
        df.loc[mask, 'port_long'] = coords['longitude']
    
    return df

# Apply the function to train_df
train_df = add_port_coordinates(train_df, ports_df)

# Verify the results
print("\nSample of updated training data:")
print(train_df[['portId', 'port_lat', 'port_long']].head())

# Check for any rows where mapping failed
unmapped = train_df[train_df['port_lat'].isna()]['portId'].nunique()
print(f"\nNumber of unique portIds with no mapping: {unmapped}")




Sample of updated training data:
                     portId   port_lat  port_long
0  61d376b393c6feb83e5eb50c  13.263333  80.341111
1  61d376d893c6feb83e5eb546  18.941944  72.885278
2  61d376d893c6feb83e5eb546  18.941944  72.885278
3  61d376d893c6feb83e5eb546  18.941944  72.885278
4  61d376d893c6feb83e5eb546  18.941944  72.885278

Number of unique portIds with no mapping: 0


In [94]:
train_df.head()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,port_lat,port_long
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,1,1,0,0,0,0,0,0,13.263333,80.341111
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,1,1,0,0,0,0,0,0,18.941944,72.885278
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,1,1,0,0,0,0,0,0,18.941944,72.885278
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,1,1,0,0,0,0,0,0,18.941944,72.885278
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,1,0,0,0,0,0,0,0,18.941944,72.885278


In [95]:
test_df.columns

Index(['ID', 'vesselId', 'time', 'scaling_factor', 'port_lat', 'port_long',
       'month_of_the_year', 'week_of_the_year', 'day_of_the_year',
       'day_of_the_month', 'day_of_the_week', 'hour_of_the_day',
       'hours_passed', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
       'time_diff_gt_10min', 'time_diff_gt_20min', 'time_diff_gt_40min',
       'time_diff_gt_1hour', 'time_diff_gt_2hours', 'time_diff_gt_6hours',
       'time_diff_gt_12hours', 'time_diff_gt_1day'],
      dtype='object')

In [96]:
port_predictor_features = ['time',
       'month_of_the_year', 'week_of_the_year', 'day_of_the_year',
       'day_of_the_month', 'day_of_the_week', 'hour_of_the_day',
       'hours_passed', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
       'time_diff_gt_10min', 'time_diff_gt_20min', 'time_diff_gt_40min',
       'time_diff_gt_1hour', 'time_diff_gt_2hours', 'time_diff_gt_6hours',
       'time_diff_gt_12hours', 'time_diff_gt_1day']

vessel_ids_lacking_port = test_df[
    (test_df['port_lat'].isna()) | 
    (test_df['port_long'].isna())
]['vesselId'].unique()

print(f"Number of vessels with missing port coordinates: {len(vessel_ids_lacking_port)}")

Number of vessels with missing port coordinates: 156


In [97]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
import numpy as np

def filter_vessel_data(vessel_data, min_samples_per_port=2):
    """
    Filter vessel data to ensure minimum samples per port
    """
    port_counts = vessel_data['portId'].value_counts()
    valid_ports = port_counts[port_counts >= min_samples_per_port].index
    return vessel_data[vessel_data['portId'].isin(valid_ports)]

def train_port_classifier(vessel_id, train_df, features, min_samples_per_port=2):
    """
    Train a port classifier for a specific vessel
    """
    # Filter data for this vessel
    vessel_data = train_df[train_df['vesselId'] == vessel_id].copy()
    
    # Skip if no data or no valid port IDs
    if len(vessel_data) == 0 or vessel_data['portId'].isna().all():
        return None, None, {'error': 'No valid data'}
    
    # Remove rows with NaN port IDs
    vessel_data = vessel_data.dropna(subset=['portId'])
    
    # Filter to ensure minimum samples per port
    vessel_data = filter_vessel_data(vessel_data, min_samples_per_port)
    
    # Get unique ports for this vessel
    unique_ports = vessel_data['portId'].unique()
    
    # If fewer than 2 ports remain after filtering, skip training
    if len(unique_ports) < 2:
        return None, unique_ports, {'error': f'Unique port: {unique_ports}'}
    
    # Prepare features and target
    X = vessel_data[features]
    y = vessel_data['portId']
    
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Initialize model
    classifier = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    )
    
    # Train model
    classifier.fit(X_train, y_train)
    
    # Make predictions on validation set
    y_pred = classifier.predict(X_val)
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'training_samples': len(X_train),
        'unique_ports': len(unique_ports),
    }
    
    return classifier, unique_ports, metrics
        

# Dictionary to store models for each vessel
vessel_classifiers = {}

# Train classifiers for each vessel with sufficient data
MIN_SAMPLES_PER_PORT = 2

for vessel_id in vessel_ids_lacking_port:
    classifier, unique_ports, metrics = train_port_classifier(
        vessel_id, train_df, port_predictor_features, 
        min_samples_per_port=MIN_SAMPLES_PER_PORT
    )
    
    if metrics is not None and 'error' not in metrics:
        vessel_classifiers[vessel_id] = {
            'classifier': classifier,
            #'unique_ports': unique_ports,
            'metrics': metrics
        }

# Print summary
print(f"\nSuccessfully trained classifiers for {len(vessel_classifiers)} vessels")
print(f"Failed to train classifiers for {len(vessel_ids_lacking_port) - len(vessel_classifiers)} vessels")


Successfully trained classifiers for 154 vessels
Failed to train classifiers for 2 vessels


In [98]:
# Extract accuracies from all classifiers
accuracies = [model['metrics']['accuracy'] for model in vessel_classifiers.values()]

# Calculate statistics
mean_acc = np.mean(accuracies)
median_acc = np.median(accuracies)
min_acc = np.min(accuracies)
max_acc = np.max(accuracies)

print(f"Classifier Performance Statistics:")
print(f"Mean accuracy: {mean_acc:.3f}")
print(f"Median accuracy: {median_acc:.3f}")
print(f"Min accuracy: {min_acc:.3f}")
print(f"Max accuracy: {max_acc:.3f}")

Classifier Performance Statistics:
Mean accuracy: 0.947
Median accuracy: 0.959
Min accuracy: 0.630
Max accuracy: 0.999


In [99]:
def fill_missing_port_coordinates(test_df, vessel_classifiers, ports_df):
    """
    Fill missing port coordinates in test_df using trained classifiers or unique ports
    """
    # Create a copy of test_df to avoid modifying the original
    df = test_df.copy()
    
    # Create a mapping of portId to coordinates
    port_coords = ports_df.set_index('portId')[['latitude', 'longitude']].to_dict('index')
    
    # Identify rows with missing coordinates
    missing_coords_mask = df['port_lat'].isna() | df['port_long'].isna()
    
    # Process each vessel's data
    for vessel_id in df[missing_coords_mask]['vesselId'].unique():
        # Get rows for this vessel that need predictions
        vessel_mask = (df['vesselId'] == vessel_id) & missing_coords_mask
        vessel_rows = df[vessel_mask]
        
        if len(vessel_rows) == 0:
            continue
            
        # Check if we have a classifier for this vessel
        vessel_data = train_port_classifier(vessel_id, train_df, port_predictor_features)
        
        if vessel_data[0] is None:  # No classifier, use unique port
            if vessel_data[1] is not None and len(vessel_data[1]) > 0:
                predicted_port = vessel_data[1][0]  # Take the first (only) unique port
                
                # Fill coordinates for this port
                if predicted_port in port_coords:
                    df.loc[vessel_mask, 'port_lat'] = port_coords[predicted_port]['latitude']
                    df.loc[vessel_mask, 'port_long'] = port_coords[predicted_port]['longitude']
        
        else:  # Use classifier
            classifier = vessel_classifiers[vessel_id]['classifier']
            
            # Get features for prediction
            X_pred = vessel_rows[port_predictor_features]
            
            # Make predictions
            predicted_ports = classifier.predict(X_pred)
            
            # Fill coordinates for each prediction
            for i, (idx, row) in enumerate(vessel_rows.iterrows()):
                predicted_port = predicted_ports[i]
                if predicted_port in port_coords:
                    df.loc[idx, 'port_lat'] = port_coords[predicted_port]['latitude']
                    df.loc[idx, 'port_long'] = port_coords[predicted_port]['longitude']
    
    return df

# Apply the function
test_df_filled = fill_missing_port_coordinates(test_df, vessel_classifiers, ports_df)

# Print summary of changes
original_nulls = test_df['port_lat'].isna().sum() + test_df['port_long'].isna().sum()
remaining_nulls = test_df_filled['port_lat'].isna().sum() + test_df_filled['port_long'].isna().sum()

print(f"\nOriginal missing coordinates: {original_nulls}")
print(f"Remaining missing coordinates: {remaining_nulls}")
print(f"Filled {original_nulls - remaining_nulls} coordinates")

test_df = test_df_filled


Original missing coordinates: 73836
Remaining missing coordinates: 0
Filled 73836 coordinates


In [100]:
test_df.head(50)

,ID,vesselId,time,scaling_factor,port_lat,port_long,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
0,4,61e9f38eb937134a3c4bfd8d,0.349750,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.975528,0.654508,1,1,0,0,0,0,0,0
1,201,61e9f38eb937134a3c4bfd8d,0.349802,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.206107,1,1,1,0,0,0,0,0
2,583,61e9f38eb937134a3c4bfd8d,0.349904,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.345492,0.024472,1,0,0,0,0,0,0,0
3,701,61e9f38eb937134a3c4bfd8d,0.349938,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.793893,1,0,0,0,0,0,0,0
4,829,61e9f38eb937134a3c4bfd8d,0.349961,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.654508,0.975528,1,1,0,0,0,0,0,0
5,1038,61e9f38eb937134a3c4bfd8d,0.350024,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.206107,0.095492,1,0,0,0,0,0,0,0
6,1114,61e9f38eb937134a3c4bfd8d,0.350052,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.793893,1,1,0,0,0,0,0,0
7,1258,61e9f38eb937134a3c4bfd8d,0.350092,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.975528,0.654508,0,0,0,0,0,0,0,0
8,1396,61e9f38eb937134a3c4bfd8d,0.350109,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.904508,0.206107,1,1,0,0,0,0,0,0
9,1540,61e9f38eb937134a3c4bfd8d,0.350166,0.30,48.380556,-4.474167,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.793893,1,1,0,0,0,0,0,0


In [101]:
def add_port_distance_features(df):
    # Distance from last known position to port
    df['lat_diff_to_port_1step'] = df['latitude_1_steps_ago'] - df['port_lat']
    df['long_diff_to_port_1step'] = df['longitude_1_steps_ago'] - df['port_long']
    
    # Absolute distances
    df['abs_lat_diff_to_port_1step'] = abs(df['lat_diff_to_port_1step'])
    df['abs_long_diff_to_port_1step'] = abs(df['long_diff_to_port_1step'])
    
    # Rate of change in distance to port (handle division by zero)
    lat_diff = df['lat_diff_to_port_1step']
    long_diff = df['long_diff_to_port_1step']
    
    df['lat_approach_rate'] = (df['latitude_1_steps_ago'] - df['latitude_2_steps_ago'])
    df['lat_approach_rate'] = np.where(lat_diff != 0, 
                                      df['lat_approach_rate'] / lat_diff, 
                                      0)  # Set to 0 when denominator is 0
    
    df['long_approach_rate'] = (df['longitude_1_steps_ago'] - df['longitude_2_steps_ago'])
    df['long_approach_rate'] = np.where(long_diff != 0, 
                                       df['long_approach_rate'] / long_diff, 
                                       0)  # Set to 0 when denominator is 0
    
    # Direction features
    df['moving_towards_port_lat'] = ((abs(df['latitude_2_steps_ago'] - df['port_lat']) - 
                                    abs(df['latitude_1_steps_ago'] - df['port_lat'])) > 0).astype(int)
    df['moving_towards_port_long'] = ((abs(df['longitude_2_steps_ago'] - df['port_long']) - 
                                     abs(df['longitude_1_steps_ago'] - df['port_long'])) > 0).astype(int)
    
    # Euclidean distance (add small epsilon to avoid zero)
    df['euclidean_dist_to_port'] = np.sqrt(
        df['lat_diff_to_port_1step']**2 + 
        df['long_diff_to_port_1step']**2 + 
        1e-10  # Small constant to avoid zero
    )
    
    # Clip any extreme values
    for col in ['lat_approach_rate', 'long_approach_rate',]:
        df[col] = df[col].clip(-1e6, 1e6)  # Clip to reasonable range
        
    # Replace any remaining infinities with 0
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

add_port_distance_features(train_df)
train_df.head()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,port_long,lat_diff_to_port_1step,long_diff_to_port_1step,abs_lat_diff_to_port_1step,abs_long_diff_to_port_1step,lat_approach_rate,long_approach_rate,moving_towards_port_lat,moving_towards_port_long,euclidean_dist_to_port
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,80.341111,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.438334,4.698122,11.438334,4.698122,NaN,NaN,0,0,12.365591
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.368924,4.609772,11.368924,4.609772,-0.006105,-0.019166,1,1,12.267943
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.291514,4.508762,11.291514,4.508762,-0.006856,-0.022403,1,1,12.158422
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,72.885278,-11.229194,4.428662,11.229194,4.428662,-0.005550,-0.018087,1,1,12.070950


In [102]:
train_df.to_csv('../data/processed_data/train.csv', index=False)
test_df.to_csv("../data/processed_data/test.csv", index=False)